# 05 — Warm-start protocols (claim F6)
Train × test matrix over P0, P1, P1b, P2 (3 seeds × 20k each, production conditions). Expected: much weaker
effects than System I (slowest closed-loop time constant ~12 min vs a 300-min history).

**PRE-REGISTER (fill and date before running):** `rules.protocol_rule` on the P2 test column (System I nb29
rule); production stays P2 unless the rule selects otherwise (a user decision then).

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
from system_iii import inference as inf
import pickle
PROTS = ["P0", "P1", "P1b", "P2"]
banks, tests = {}, {}
for p in PROTS:
    f = DATA / f"05_bank_{p}.npz"
    if not f.exists():
        da.save_npz(f, da.make_windows(sim.prior_draws(60_000, 5100), 5100, protocol=p))
    banks[p] = da.load_npz(f)
    g = DATA / f"05_test_{p}.npz"
    if not g.exists():
        da.save_npz(g, da.test_set(5199, protocol=p))
    tests[p] = da.load_npz(g)
rows = []
for p in PROTS:
    for s in range(3):
        f = R / "05" / f"post_{p}_s{s}.pkl"; f.parent.mkdir(exist_ok=True)
        if f.exists():
            post = pickle.load(open(f, "rb"))
        else:
            sl = slice(s * 20_000, (s + 1) * 20_000)
            post, _ = inf.train_npe(banks[p]["theta"][sl], banks[p]["x"][sl], seed=s); pickle.dump(post, open(f, "wb"))
        for q in PROTS:
            row, _ = inf.evaluate(post, tests[q], seed=s)
            rows.append({"train": p, "test": q, "seed": s, **row})
mat = pd.DataFrame(rows); mat.to_csv(R / "05_protocol_matrix.csv", index=False)
dec = rules.protocol_rule(mat, test_col="P2"); print(dec["choice"], dec["flag"])
da.save_json(R / "05_protocol_decision.json", dec)
display(mat.pivot_table(index="train", columns="test", values="cov90_b").round(3))

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).